# <mark>**Startport**</mark>

In [1]:
# =========================
# STEP RUNNER (NO AUTO-EXEC)
# =========================

import traceback

STEP_RESULTS = []

def run_step(name, fn):
    print(f"\n▶️ Running {name}...")
    try:
        fn()
        STEP_RESULTS.append((name, "SUCCESS", None))
        print(f"✅ {name} completed")
    except Exception as e:
        STEP_RESULTS.append((name, "FAILED", str(e)))
        print(f"❌ {name} failed:")
        print(traceback.format_exc())


StatementMeta(, 965c8fe1-ad75-4d47-803c-efabd09ce546, 3, Finished, Available, Finished, False)

# <mark>**SFTP ➜ SharePoint**</mark>

In [2]:
def step1():
    # ---- Graph / SharePoint upload target ----
    global TENANT_ID, CLIENT_ID, CLIENT_SECRET
    TENANT_ID     = "9c5ff121-707c-46c0-a4a1-79341c78f055"
    CLIENT_ID     = "cd2f664c-c455-4d56-8359-0550c71b4da9"
    CLIENT_SECRET = "YOUR_SHAREPOINT_CLIENT_SECRET_HERE"

    # ---- SharePoint destination ----
    global SHAREPOINT_HOSTNAME, DOC_LIBRARY_FOLDER_PATH
    SHAREPOINT_HOSTNAME = "heritagehillpm.sharepoint.com"
    DOC_LIBRARY_FOLDER_PATH = "Fabric Data Drop/OneSite Reports"

    # ---- SFTP source ----
    global SFTP_HOST, SFTP_PORT, SFTP_USER, SFTP_PASS
    SFTP_HOST = "emft.realpage.com"
    SFTP_PORT = 22
    SFTP_USER = "EXT-3712076_OS"
    SFTP_PASS = "7QV08u8KHLWtnU"

    # Folders:
    global SFTP_PRIMARY_FOLDER, SFTP_SECONDARY_FOLDER
    SFTP_PRIMARY_FOLDER   = "/os_home/ReportGroups/Fabric Reports"     # proceed/fail evaluated here ONLY
    SFTP_SECONDARY_FOLDER = "/os_home/ReportGroups/Fabric Reports 2"   # INCLUDED for upload payload

    # ---- Behavior flags ----
    global DRY_RUN, OVERWRITE, CLEAR_SHAREPOINT_FOLDER
    global KEEP_LOCAL_ZIPS, KEEP_LOCAL_EXTRACTED, UPLOAD_ONLY_EXTENSIONS
    DRY_RUN = False                  # True = do not download/unzip/delete/upload; just show matches
    OVERWRITE = True                 # True = replace same-name files in SharePoint
    CLEAR_SHAREPOINT_FOLDER = True   # True = DELETE ALL files currently in the SharePoint folder before upload
    KEEP_LOCAL_ZIPS = False          # True = keep downloaded zip(s) locally after run
    KEEP_LOCAL_EXTRACTED = False     # True = keep extracted files locally after run
    UPLOAD_ONLY_EXTENSIONS = None    # e.g. {".xlsx", ".csv"} or None for all extracted files

    # Local staging folders (driver local FS)
    global LOCAL_WORK_DIR, LOCAL_ZIP_DIR, LOCAL_EXTRACT_DIR
    LOCAL_WORK_DIR = "/tmp/realpage_sftp_pull"
    LOCAL_ZIP_DIR  = f"{LOCAL_WORK_DIR}/zips"
    LOCAL_EXTRACT_DIR = f"{LOCAL_WORK_DIR}/extracted"

StatementMeta(, 965c8fe1-ad75-4d47-803c-efabd09ce546, 4, Finished, Available, Finished, False)

In [3]:
def step2():
    # ============================================================
    # Fabric PySpark Notebook (Driver-side Python):
    #
    # GOAL (same functionality):
    #   1) PROCEED/FAIL evaluation based ONLY on PRIMARY folder:
    #        /os_home/ReportGroups/Fabric Reports
    #      ...and ONLY if it contains TODAY's:
    #        SpecificDate_SystemDate_<M>_<D>_<YYYY>*.zip
    #
    #   2) Upload payload includes BOTH folders:
    #        - PRIMARY: SpecificDate_SystemDate_<M>_<D>_<YYYY>*.zip (TODAY)
    #        - SECONDARY: EITHER naming style for TODAY:
    #            a) OnDemand_*<YYYYMMDD>*.zip
    #            b) SpecificDate_SystemDate_<M>_<D>_<YYYY>*.zip
    #
    #   3) Download -> unzip -> CLEAR SharePoint folder -> upload extracted files
    #
    # HARDENING ADDED (no functional change):
    #   - Dedup extracted files by filename before upload (prevents double-upload conflicts)
    #   - Confirm SharePoint folder is empty after delete (wait/poll)
    #   - Retry uploads on 409/5xx/429; if small PUT conflicts, fallback to upload session (replace)
    # ============================================================

    # ==============================
    # CONFIG
    # ==============================

    # ---- Graph / SharePoint upload target ----
    TENANT_ID     = "9c5ff121-707c-46c0-a4a1-79341c78f055"
    CLIENT_ID     = "cd2f664c-c455-4d56-8359-0550c71b4da9"
    CLIENT_SECRET = "YOUR_SHAREPOINT_CLIENT_SECRET_HERE"

    # ---- SharePoint destination ----
    SHAREPOINT_HOSTNAME = "heritagehillpm.sharepoint.com"
    DOC_LIBRARY_FOLDER_PATH = "Fabric Data Drop/OneSite Reports"

    # ---- SFTP source ----
    SFTP_HOST = "emft.realpage.com"
    SFTP_PORT = 22
    SFTP_USER = "EXT-3712076_OS"
    SFTP_PASS = "7QV08u8KHLWtnU"

    # Folders:
    SFTP_PRIMARY_FOLDER   = "/os_home/ReportGroups/Fabric Reports"     # proceed/fail evaluated here ONLY
    SFTP_SECONDARY_FOLDER = "/os_home/ReportGroups/Fabric Reports 2"   # INCLUDED for upload payload

    # ---- Behavior flags ----
    DRY_RUN = False                  # True = do not download/unzip/delete/upload; just show matches
    OVERWRITE = True                 # True = replace same-name files in SharePoint
    CLEAR_SHAREPOINT_FOLDER = True   # True = DELETE ALL files currently in the SharePoint folder before upload
    KEEP_LOCAL_ZIPS = False          # True = keep downloaded zip(s) locally after run
    KEEP_LOCAL_EXTRACTED = False     # True = keep extracted files locally after run
    UPLOAD_ONLY_EXTENSIONS = None    # e.g. {".xlsx", ".csv"} or None for all extracted files

    # Local staging folders (driver local FS)
    LOCAL_WORK_DIR = "/tmp/realpage_sftp_pull"
    LOCAL_ZIP_DIR  = f"{LOCAL_WORK_DIR}/zips"
    LOCAL_EXTRACT_DIR = f"{LOCAL_WORK_DIR}/extracted"

    # ==============================
    # Imports
    # ==============================
    import os, re, urllib.parse, shutil, zipfile, time
    from datetime import datetime
    import requests

    from msal import ConfidentialClientApplication
    import paramiko

    # ==============================
    # Helpers
    # ==============================
    def ensure_dir(path: str):
        os.makedirs(path, exist_ok=True)

    def clean_dir(path: str):
        if os.path.exists(path):
            shutil.rmtree(path)
        os.makedirs(path, exist_ok=True)

    def today_components():
        now = datetime.now()
        return now.month, now.day, now.year

    def build_primary_regex_for_today():
        m, d, y = today_components()
        prefix = f"SpecificDate_SystemDate_{m}_{d}_{y}"
        rx = re.compile(rf"^{re.escape(prefix)}.*\.zip$", re.IGNORECASE)
        return prefix, rx

    def build_secondary_regex_for_today():
        m, d, y = today_components()
        yyyymmdd = f"{y}{m:02d}{d:02d}"

        ondemand_rx = rf"OnDemand_.*{yyyymmdd}.*\.zip"
        specific_prefix = f"SpecificDate_SystemDate_{m}_{d}_{y}"
        specific_rx = rf"{re.escape(specific_prefix)}.*\.zip"

        combined = re.compile(rf"^(?:{ondemand_rx}|{specific_rx})$", re.IGNORECASE)
        return yyyymmdd, specific_prefix, combined

    def graph_token():
        app = ConfidentialClientApplication(
            CLIENT_ID,
            authority=f"https://login.microsoftonline.com/{TENANT_ID}",
            client_credential=CLIENT_SECRET
        )
        tok = app.acquire_token_for_client(scopes=["https://graph.microsoft.com/.default"])
        assert "access_token" in tok, f"Token acquisition failed: {tok}"
        return tok["access_token"]

    # ------------------------------
    # SharePoint resolve (SITE + PATH)
    # ------------------------------
    def resolve_root_site_id(hdr: dict) -> str:
        r = requests.get(f"https://graph.microsoft.com/v1.0/sites/{SHAREPOINT_HOSTNAME}:/", headers=hdr)
        r.raise_for_status()
        return r.json()["id"]

    def resolve_default_drive_id(site_id: str, hdr: dict) -> str:
        r = requests.get(f"https://graph.microsoft.com/v1.0/sites/{site_id}/drive", headers=hdr)
        r.raise_for_status()
        return r.json()["id"]

    def resolve_folder_item_id(drive_id: str, folder_path: str, hdr: dict) -> str:
        url = f"https://graph.microsoft.com/v1.0/drives/{drive_id}/root:/{urllib.parse.quote(folder_path)}"
        r = requests.get(url, headers=hdr)
        r.raise_for_status()
        return r.json()["id"]

    # ------------------------------
    # SharePoint folder cleanup
    # ------------------------------
    def sp_list_children(drive_id: str, folder_item_id: str, hdr: dict):
        url = f"https://graph.microsoft.com/v1.0/drives/{drive_id}/items/{folder_item_id}/children?$top=999"
        out = []
        while url:
            r = requests.get(url, headers=hdr)
            if not r.ok:
                print("\nLIST CHILDREN FAILED")
                print("Status:", r.status_code)
                print("Response:", r.text)
                r.raise_for_status()
            j = r.json()
            out.extend(j.get("value", []))
            url = j.get("@odata.nextLink")
        return out

    def sp_delete_item(drive_id: str, item_id: str, hdr: dict):
        url = f"https://graph.microsoft.com/v1.0/drives/{drive_id}/items/{item_id}"
        r = requests.delete(url, headers=hdr)
        if r.status_code in (204, 404):
            return True
        print("\nDELETE FAILED")
        print("Status:", r.status_code)
        print("Response:", r.text)
        r.raise_for_status()

    def sp_clear_folder_files(drive_id: str, folder_item_id: str, hdr: dict):
        children = sp_list_children(drive_id, folder_item_id, hdr)
        files = [c for c in children if c.get("file")]
        folders = [c for c in children if c.get("folder")]

        print(f"\nSharePoint folder currently has {len(files)} file(s) and {len(folders)} subfolder(s).")
        if len(files) == 0:
            print("Nothing to delete.")
            return

        print("Deleting existing files in SharePoint folder...")
        for f in files:
            name = f.get("name", "unknown")
            item_id = f.get("id")
            sp_delete_item(drive_id, item_id, hdr)
            print(f"  Deleted: {name}")

        print(f"Deleted {len(files)} file(s) from SharePoint folder.")

    def sp_wait_until_empty(drive_id: str, folder_item_id: str, hdr: dict, timeout_sec=60, poll_sec=2):
        """
        After deletes, SharePoint can take a moment to become consistent.
        This avoids upload racing with deletions.
        """
        start = time.time()
        while True:
            children = sp_list_children(drive_id, folder_item_id, hdr)
            files = [c for c in children if c.get("file")]
            if len(files) == 0:
                return True
            if time.time() - start > timeout_sec:
                print(f"⚠️ Folder not empty after {timeout_sec}s. Continuing anyway ({len(files)} file(s) still listed).")
                return False
            time.sleep(poll_sec)

    # ------------------------------
    # Upload helpers (hardened)
    # ------------------------------
    def _req_with_retry(method, url, *, headers=None, data=None, json=None, ok_statuses=(200,201,202,204), tries=6, base_sleep=1.0):
        """
        Retries on 409 (resourceModified), 429, and 5xx.
        """
        last = None
        for i in range(tries):
            r = requests.request(method, url, headers=headers, data=data, json=json)
            if r.status_code in ok_statuses:
                return r

            if r.status_code in (409, 429) or (500 <= r.status_code <= 599):
                wait = base_sleep * (2 ** i)
                ra = r.headers.get("Retry-After")
                if ra:
                    try:
                        wait = max(wait, float(ra))
                    except:
                        pass
                time.sleep(min(wait, 30))
                last = r
                continue

            r.raise_for_status()

        if last is not None:
            print("\nREQUEST FAILED AFTER RETRIES")
            print("Status:", last.status_code)
            print("Response:", last.text)
            last.raise_for_status()
        raise Exception("Request failed unexpectedly")

    def sp_upload_small_put(drive_id: str, folder_item_id: str, filename: str, content_bytes: bytes, hdr: dict):
        safe_name = filename.replace("/", "_").replace("\\", "_")
        url = f"https://graph.microsoft.com/v1.0/drives/{drive_id}/items/{folder_item_id}:/{urllib.parse.quote(safe_name)}:/content"
        r = _req_with_retry(
            "PUT",
            url,
            headers={**hdr, "Content-Type": "application/octet-stream"},
            data=content_bytes,
            ok_statuses=(200, 201)
        )
        return r.json()

    def sp_upload_large_session(drive_id: str, folder_item_id: str, filename: str, filepath: str, hdr: dict, chunk_size=10*1024*1024):
        safe_name = filename.replace("/", "_").replace("\\", "_")
        create_url = f"https://graph.microsoft.com/v1.0/drives/{drive_id}/items/{folder_item_id}:/{urllib.parse.quote(safe_name)}:/createUploadSession"

        body = {
            "item": {
                "@microsoft.graph.conflictBehavior": ("replace" if OVERWRITE else "rename"),
                "name": safe_name
            }
        }

        r = _req_with_retry(
            "POST",
            create_url,
            headers={**hdr, "Content-Type": "application/json"},
            json=body,
            ok_statuses=(200, 201)
        )

        upload_url = r.json()["uploadUrl"]
        total_size = os.path.getsize(filepath)
        sent = 0

        with open(filepath, "rb") as f:
            while sent < total_size:
                chunk = f.read(chunk_size)
                if not chunk:
                    break

                start = sent
                end = sent + len(chunk) - 1
                headers = {
                    "Content-Length": str(len(chunk)),
                    "Content-Range": f"bytes {start}-{end}/{total_size}"
                }

                put = requests.put(upload_url, headers=headers, data=chunk)
                if put.status_code not in (200, 201, 202):
                    ok = False
                    for i in range(5):
                        time.sleep(min(2 ** i, 10))
                        put = requests.put(upload_url, headers=headers, data=chunk)
                        if put.status_code in (200, 201, 202):
                            ok = True
                            break
                    if not ok:
                        print("\nCHUNK UPLOAD FAILED")
                        print("Status:", put.status_code)
                        print("File:", safe_name)
                        print("Response:", put.text)
                        raise Exception(f"Chunk upload failed {put.status_code}")

                sent += len(chunk)

        return True

    def sp_upload_any(drive_id: str, folder_item_id: str, fname: str, fp: str, hdr: dict):
        """
        Keeps original behavior:
          - small files: PUT /content
          - large files: upload session
        Hardening:
          - if small PUT keeps conflicting, fallback to upload session (replace)
        """
        size = os.path.getsize(fp)

        if size <= 3.5 * 1024 * 1024:
            with open(fp, "rb") as f:
                content = f.read()
            try:
                sp_upload_small_put(drive_id, folder_item_id, fname, content, hdr)
            except requests.HTTPError as e:
                resp = getattr(e, "response", None)
                if resp is not None and resp.status_code == 409:
                    sp_upload_large_session(drive_id, folder_item_id, fname, fp, hdr)
                else:
                    raise
        else:
            sp_upload_large_session(drive_id, folder_item_id, fname, fp, hdr)

        return size

    # ------------------------------
    # SFTP
    # ------------------------------
    def sftp_list_matching_zips(sftp: paramiko.SFTPClient, folder: str, rx):
        matches = []
        try:
            for fname in sftp.listdir(folder):
                if rx.match(fname):
                    matches.append((folder, fname))
        except FileNotFoundError:
            pass
        return matches

    def sftp_download(sftp: paramiko.SFTPClient, remote_folder: str, remote_name: str, local_path: str):
        remote_full = f"{remote_folder.rstrip('/')}/{remote_name}"
        sftp.get(remote_full, local_path)

    # ------------------------------
    # Zip extract
    # ------------------------------
    def unzip_to_dir(zip_path: str, extract_dir: str):
        extracted_files = []
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(extract_dir)

        for root, _, files in os.walk(extract_dir):
            for f in files:
                extracted_files.append(os.path.join(root, f))

        return extracted_files

    def filter_extracted(files, allowed_exts):
        if not allowed_exts:
            return files
        allowed_exts = {e.lower() for e in allowed_exts}
        out = []
        for fp in files:
            ext = os.path.splitext(fp)[1].lower()
            if ext in allowed_exts:
                out.append(fp)
        return out

    def dedup_files_by_name(file_paths):
        """
        Prevents duplicate upload attempts for same filename (common when two zips include same report).
        Keeps the LAST seen path (usually later zip processed).
        """
        m = {}
        for p in file_paths:
            m[os.path.basename(p)] = p
        return [m[k] for k in sorted(m.keys())]

    # ==============================
    # MAIN
    # ==============================
    primary_prefix, primary_rx = build_primary_regex_for_today()
    secondary_yyyymmdd, secondary_specific_prefix, secondary_rx = build_secondary_regex_for_today()

    print(f"PRIMARY required pattern: {primary_prefix}*.zip")
    print("SECONDARY required patterns (today):")
    print(f" - OnDemand_*{secondary_yyyymmdd}*.zip")
    print(f" - {secondary_specific_prefix}*.zip")
    print(f"ENFORCED proceed/fail folder: {SFTP_PRIMARY_FOLDER}")
    print("Upload will include PRIMARY + SECONDARY.")

    clean_dir(LOCAL_WORK_DIR)
    ensure_dir(LOCAL_ZIP_DIR)
    ensure_dir(LOCAL_EXTRACT_DIR)

    transport = paramiko.Transport((SFTP_HOST, SFTP_PORT))
    transport.connect(username=SFTP_USER, password=SFTP_PASS)
    sftp = paramiko.SFTPClient.from_transport(transport)

    primary_matches = sftp_list_matching_zips(sftp, SFTP_PRIMARY_FOLDER, primary_rx)

    if not primary_matches:
        try:
            print("\nDEBUG: ZIPs currently in PRIMARY folder:")
            for f in sftp.listdir(SFTP_PRIMARY_FOLDER):
                if f.lower().endswith(".zip"):
                    print(" -", f)
        except Exception as e:
            print("DEBUG listdir failed:", repr(e))

        sftp.close()
        transport.close()

        raise Exception(
            "FAILURE: PRIMARY SFTP folder does NOT contain TODAY's SpecificDate_SystemDate zip.\n\n"
            f"PRIMARY folder:\n  {SFTP_PRIMARY_FOLDER}\n\n"
            f"Expected pattern:\n  {primary_prefix}*.zip\n\n"
            "Action taken:\n  Notebook aborted BEFORE clearing SharePoint or uploading anything.\n"
        )

    upload_matches = []
    upload_matches.extend(primary_matches)

    secondary_matches = sftp_list_matching_zips(sftp, SFTP_SECONDARY_FOLDER, secondary_rx)
    upload_matches.extend(secondary_matches)

    seen = set()
    dedup = []
    for rf, rn in upload_matches:
        key = f"{rf.rstrip('/')}/{rn}"
        if key not in seen:
            seen.add(key)
            dedup.append((rf, rn))
    upload_matches = dedup

    print("\nTODAY zips found for upload:")
    for rf, rn in upload_matches:
        print(f" - {rf}/{rn}")

    if DRY_RUN:
        print("\nDRY_RUN=True: stopping before download/unzip/delete/upload.")
        sftp.close()
        transport.close()
        raise SystemExit("DRY_RUN complete.")

    token = graph_token()
    hdr = {"Authorization": f"Bearer {token}"}

    site_id = resolve_root_site_id(hdr)
    drive_id = resolve_default_drive_id(site_id, hdr)
    folder_item_id = resolve_folder_item_id(drive_id, DOC_LIBRARY_FOLDER_PATH, hdr)

    print("\nResolved SharePoint target (site+path):")
    print(" - siteId:", site_id)
    print(" - driveId:", drive_id)
    print(" - folderItemId:", folder_item_id)
    print(" - folderPath:", DOC_LIBRARY_FOLDER_PATH)

    zip_processed = 0
    extracted_files_all = []

    for remote_folder, remote_name in upload_matches:
        zip_processed += 1
        local_zip = os.path.join(LOCAL_ZIP_DIR, remote_name)

        print(f"\nDownloading: {remote_folder}/{remote_name}")
        sftp_download(sftp, remote_folder, remote_name, local_zip)

        extract_subdir = os.path.join(LOCAL_EXTRACT_DIR, os.path.splitext(remote_name)[0])
        ensure_dir(extract_subdir)

        print(f"Extracting: {local_zip}")
        extracted_files = unzip_to_dir(local_zip, extract_subdir)
        extracted_files = filter_extracted(extracted_files, UPLOAD_ONLY_EXTENSIONS)

        applicant_hits = [os.path.basename(x) for x in extracted_files if "Applicant" in os.path.basename(x)]
        if applicant_hits:
            print("  Applicant-related extracted files:", applicant_hits)

        extracted_files_all.extend(extracted_files)

        if not KEEP_LOCAL_ZIPS:
            try:
                os.remove(local_zip)
            except:
                pass

    sftp.close()
    transport.close()

    if not extracted_files_all:
        raise Exception("\nFAILURE: Zips extracted 0 files after filtering. Nothing to upload.")

    before = len(extracted_files_all)
    extracted_files_all = dedup_files_by_name(extracted_files_all)
    after = len(extracted_files_all)
    if after != before:
        print(f"\nDedup extracted files by filename: {before} -> {after}")

    if CLEAR_SHAREPOINT_FOLDER:
        sp_clear_folder_files(drive_id, folder_item_id, hdr)
        sp_wait_until_empty(drive_id, folder_item_id, hdr, timeout_sec=60, poll_sec=2)

    print(f"\nUploading {len(extracted_files_all)} extracted file(s) to SharePoint...")

    files_uploaded = 0
    for fp in extracted_files_all:
        fname = os.path.basename(fp)
        size = sp_upload_any(drive_id, folder_item_id, fname, fp, hdr)
        files_uploaded += 1
        print(f"  Uploaded: {fname} ({size/1024/1024:.2f} MB)")

    if not KEEP_LOCAL_EXTRACTED:
        try:
            shutil.rmtree(LOCAL_EXTRACT_DIR)
        except:
            pass

    print("\nDONE:")
    print(f" - Zips processed: {zip_processed}")
    print(f" - Files uploaded: {files_uploaded}")

StatementMeta(, 965c8fe1-ad75-4d47-803c-efabd09ce546, 5, Finished, Available, Finished, False)

# <mark>**SharePoint ➜ Lakehouse, 1file per Sheet**</mark>

In [4]:
def step3():
    # ============================================================
    # NEW CELL: SharePoint -> split Excel files by sheet -> write to Lakehouse Files/
    #   Reads files from:  Shared Documents/Fabric Data Drop/OneSite Reports
    #   Writes to:        /lakehouse/default/Files/xlsx_by_sheet
    #
    # Packages needed in Notebook Environment:
    #   - msal
    #   - requests
    #   - pandas
    #   - openpyxl
    #   - xlrd==2.0.1   (only if you expect .xls)
    #   - pyxlsb        (only if you expect .xlsb)
    # ============================================================

    # ====== EDIT THESE (if desired) ======
    global TENANT_ID, CLIENT_ID, CLIENT_SECRET
    TENANT_ID     = "9c5ff121-707c-46c0-a4a1-79341c78f055"
    CLIENT_ID     = "cd2f664c-c455-4d56-8359-0550c71b4da9"
    CLIENT_SECRET = "YOUR_SHAREPOINT_CLIENT_SECRET_HERE"

    global SHAREPOINT_HOSTNAME, DOC_LIBRARY_FOLDER_PATH
    SHAREPOINT_HOSTNAME = "heritagehillpm.sharepoint.com"
    DOC_LIBRARY_FOLDER_PATH = "Fabric Data Drop/OneSite Reports"   # inside Shared Documents

    global OUTPUT_FOLDER, CLEAR_OUTPUT
    OUTPUT_FOLDER = "Files/xlsx_by_sheet"   # Lakehouse target folder
    CLEAR_OUTPUT  = True                    # delete prior .xlsx in OUTPUT_FOLDER before writing
    # =====================================

    import os, io, re, urllib.parse, requests
    import pandas as pd
    from msal import ConfidentialClientApplication
    from pandas import ExcelWriter
    from notebookutils import mssparkutils

    # -----------------------------
    # Helpers
    # -----------------------------
    def slug(s: str) -> str:
        """Safe filename segment."""
        return re.sub(r"[^A-Za-z0-9._-]+", "_", str(s)).strip("._")

    def normalize_base(filename: str) -> str:
        """
        Stable base name:
          - drop LEADING '#######_' (first 7 digits + underscore)
          - drop TRAILING '_#######' (7 digits) if present
        """
        base = os.path.splitext(filename)[0].strip()
        base = re.sub(r'^\d{7}_', '', base)     # leading 7 digits + underscore
        base = re.sub(r'_\d{7}$', '', base)     # trailing _7digits (optional)
        return base

    def graph_token() -> str:
        app = ConfidentialClientApplication(
            CLIENT_ID,
            authority=f"https://login.microsoftonline.com/{TENANT_ID}",
            client_credential=CLIENT_SECRET
        )
        tok = app.acquire_token_for_client(scopes=["https://graph.microsoft.com/.default"])
        assert "access_token" in tok, f"Token acquisition failed: {tok}"
        return tok["access_token"]

    def resolve_root_site_id(hdr: dict) -> str:
        r = requests.get(f"https://graph.microsoft.com/v1.0/sites/{SHAREPOINT_HOSTNAME}:/", headers=hdr)
        r.raise_for_status()
        return r.json()["id"]

    def resolve_default_drive_id(site_id: str, hdr: dict) -> str:
        r = requests.get(f"https://graph.microsoft.com/v1.0/sites/{site_id}/drive", headers=hdr)
        r.raise_for_status()
        return r.json()["id"]

    def resolve_folder_item_id(drive_id: str, folder_path: str, hdr: dict) -> str:
        url = f"https://graph.microsoft.com/v1.0/drives/{drive_id}/root:/{urllib.parse.quote(folder_path)}"
        r = requests.get(url, headers=hdr)
        r.raise_for_status()
        return r.json()["id"]

    def list_folder_children(drive_id: str, folder_item_id: str, hdr: dict):
        """
        Lists children of a folder driveItem (paginates).
        """
        url = f"https://graph.microsoft.com/v1.0/drives/{drive_id}/items/{folder_item_id}/children?$top=999"
        out = []
        while url:
            r = requests.get(url, headers=hdr)
            r.raise_for_status()
            j = r.json()
            out += j.get("value", [])
            url = j.get("@odata.nextLink")
        return out

    # -----------------------------
    # 1) Auth (client credentials)
    # -----------------------------
    token = graph_token()
    hdr = {"Authorization": f"Bearer {token}"}

    # -----------------------------
    # 2) Resolve SharePoint drive + folder
    # -----------------------------
    site_id = resolve_root_site_id(hdr)
    drive_id = resolve_default_drive_id(site_id, hdr)
    folder_item_id = resolve_folder_item_id(drive_id, DOC_LIBRARY_FOLDER_PATH, hdr)

    print("Resolved SharePoint source:")
    print(" - siteId:", site_id)
    print(" - driveId:", drive_id)
    print(" - folderItemId:", folder_item_id)
    print(" - folderPath:", DOC_LIBRARY_FOLDER_PATH)

    # -----------------------------
    # 3) List files in that folder
    # -----------------------------
    items = list_folder_children(drive_id, folder_item_id, hdr)

    excel_ext = {".xls", ".xlsx", ".xlsm", ".xlsb"}
    files = [
        it for it in items
        if it.get("file") and os.path.splitext(it.get("name", "").lower())[1] in excel_ext
    ]

    print(f"Found {len(items)} total item(s) in folder.")
    print(f"Found {len(files)} Excel file(s) to process.")

    # -----------------------------
    # 4) Prep output folder in Lakehouse
    # -----------------------------
    mssparkutils.fs.mkdirs(OUTPUT_FOLDER)

    if CLEAR_OUTPUT:
        try:
            for entry in mssparkutils.fs.ls(OUTPUT_FOLDER):
                if entry.name.lower().endswith(".xlsx"):
                    mssparkutils.fs.rm(entry.path)
            print(f"Cleared prior .xlsx files from: {OUTPUT_FOLDER}")
        except Exception as e:
            print("Clear output skipped/failed (non-fatal):", str(e))

    # -----------------------------
    # 5) Download each Excel file -> split by sheet -> write one xlsx per sheet
    # -----------------------------
    written = 0

    for it in files:
        name = it["name"]
        ext  = os.path.splitext(name.lower())[1]
        item_id = it["id"]

        # Download file bytes
        dl_url = f"https://graph.microsoft.com/v1.0/drives/{drive_id}/items/{item_id}/content"
        dl = requests.get(dl_url, headers=hdr)
        dl.raise_for_status()
        raw = io.BytesIO(dl.content)

        # Choose engine
        engine = "xlrd" if ext == ".xls" else ("pyxlsb" if ext == ".xlsb" else "openpyxl")

        # Enumerate sheets
        xls = pd.ExcelFile(raw, engine=engine)

        # Stable base after trimming leading/trailing 7-digit tokens
        file_base = slug(normalize_base(name))

        for sheet in xls.sheet_names:
            df = pd.read_excel(xls, sheet_name=sheet, engine=engine)

            # Skip blank sheets
            if df.dropna(how="all").empty:
                continue

            # lineage columns
            df["_source_file"] = name
            df["_sheet"] = sheet

            # output workbook path: <file_base>__<sheet>.xlsx
            sheet_base = slug(sheet) or "Sheet"
            xlsx_path  = f"/lakehouse/default/{OUTPUT_FOLDER}/{file_base}__{sheet_base}.xlsx"

            # Excel sheet name max length = 31 chars
            safe_sheet = sheet_base[:31]

            with ExcelWriter(xlsx_path, engine="openpyxl", mode="w") as xw:
                df.to_excel(xw, sheet_name=safe_sheet, index=False)

            written += 1

    print(f"DONE: Wrote {written} workbook(s) to {OUTPUT_FOLDER} (one .xlsx per file+sheet).")

StatementMeta(, 965c8fe1-ad75-4d47-803c-efabd09ce546, 6, Finished, Available, Finished, False)

# <mark>**AR ok**</mark>

In [5]:
def step4():
    # ============================================================
    # Fabric PySpark Notebook (from scratch - compatible con xlsx_by_sheet)
    #   - Lee Files/xlsx_by_sheet
    #   - Delinquent Sheet4: calcula Current y Days30Plus por Property
    #   - Resident Balances Sheet1: obtiene Current Month Billings por Property (fila Totals, col 24)
    #   - Join + calcula CurrentMonthBillingsNotCollected (%)
    #   - Guarda stg_ar_collections (overwrite)
    #   - Guarda ar_collections (append solo keys nuevas) + schema align
    # ============================================================

    from pyspark.sql import SparkSession
    from pyspark.sql import functions as F
    import pandas as pd
    from datetime import date, timedelta

    spark = SparkSession.builder.getOrCreate()

    # ------------------------------
    # CONFIG
    # ------------------------------
    FOLDER_PATH = "Files/xlsx_by_sheet"
    RUN_DATE = date.today() - timedelta(days=1)  # HOY: date.today()
    HEADER_ROW_IDX = 2
    KEY_COLS = ["RunDate", "PropertyName"]

    EXCLUDE = {
        "BBI_LLC_-_Single_Familiy_Delinquent_and_Prepaid_-_Excel__Sheet4.xlsx",
        "First_Property_I_LLC_-_Single_Family_Delinquent_and_Prepaid_-_Excel__Sheet4.xlsx",
        "Zoe_LLC_Delinquent_and_Prepaid_-_Excel__Sheet4.xlsx",
        "BBI_LLC_-_Single_Familiy_Resident_Balances_by_Fiscal_Period__Sheet1.xlsx",
        "First_Property_I_LLC_-_Single_Family_Resident_Balances_by_Fiscal_Period__Sheet1.xlsx",
        "Zoe_LLC_Resident_Balances_by_Fiscal_Period__Sheet1.xlsx",
    }

    PATTERN_DELINQ = "Delinquent_and_Prepaid_-_Excel__Sheet4"
    PATTERN_BILL   = "Resident_Balances_by_Fiscal_Period__Sheet1"

    print(f"RunDate usado: {RUN_DATE}")

    # ------------------------------
    # Helpers
    # ------------------------------
    def filename_from_path(p: str) -> str:
        return p.split("/")[-1]

    def prop_from_delinq(fname: str) -> str:
        return fname.split("_Delinquent_and_Prepaid")[0].replace("_", " ").strip()

    def prop_from_bill(fname: str) -> str:
        return fname.split("_Resident_Balances")[0].replace("_", " ").strip()

    def to_num(s: pd.Series):
        return pd.to_numeric(s, errors="coerce").fillna(0)

    def find_col(df: pd.DataFrame, options):
        """
        Busca una columna por nombre exacto o por contains (case-insensitive).
        options = ["Current", "30 Days", ...]
        """
        cols = [str(c) for c in df.columns]
        lower = [c.lower() for c in cols]

        for opt in options:
            if opt in cols:
                return opt

        for opt in options:
            opt_l = opt.lower()
            for i, c in enumerate(lower):
                if opt_l in c:
                    return cols[i]

        return None

    # =====================================================
    # 1) LIST FILES - DELINQUENT
    # =====================================================
    df_files_delinq = (
        spark.read.format("binaryFile")
        .option("pathGlobFilter", f"*{PATTERN_DELINQ}*.xlsx")
        .load(FOLDER_PATH)
        .withColumn("filename", F.element_at(F.split(F.col("path"), "/"), -1))
        .filter(~F.col("filename").isin(list(EXCLUDE)))
    )

    delinq_paths = [r[0] for r in df_files_delinq.select("path").collect()]
    print(f"Delinquent files: {len(delinq_paths)}")

    # =====================================================
    # 2) PARSE DELINQUENT (pandas) -> metrics por Property
    # =====================================================
    delinq_rows = []

    for p in delinq_paths:
        try:
            fname = filename_from_path(p)
            prop = prop_from_delinq(fname)

            df_raw = pd.read_excel(p, sheet_name=0, header=None)

            # tu lógica: salta HEADER_ROW_IDX, promote headers con la primera fila del bloque
            df = df_raw.iloc[HEADER_ROW_IDX:].copy()
            df.columns = df.iloc[0]
            df = df.iloc[1:].reset_index(drop=True)

            # normaliza headers
            df.columns = [str(c).strip() for c in df.columns]

            # filtra por columna 2 (index 1) como tu código
            if df.shape[1] < 2:
                raise Exception("No tiene suficientes columnas para filtrar por col index 1.")

            col2 = df.iloc[:, 1]
            df = df[
                col2.notna()
                & (col2.astype(str) != "2130.000")
                & (col2.astype(str) != "Delinquent/Prepaid Account")
            ]

            # quita filas donde col 0 empieza con Total
            df = df[~df.iloc[:, 0].astype(str).str.startswith("Total")]

            # detecta columnas numéricas
            c_current = find_col(df, ["Current"])
            c_30      = find_col(df, ["30 Days", "30Days"])
            c_60      = find_col(df, ["60 Days", "60Days"])
            c_90      = find_col(df, ["90+ Days", "90+Days", "90 Days", "90Days"])

            if not (c_current and c_30 and c_60 and c_90):
                raise Exception(f"Faltan columnas. Current={c_current}, 30={c_30}, 60={c_60}, 90={c_90}")

            df[c_current] = to_num(df[c_current])
            df[c_30] = to_num(df[c_30])
            df[c_60] = to_num(df[c_60])
            df[c_90] = to_num(df[c_90])

            current_sum = float(df[c_current].sum())
            plus30_sum  = float((df[c_30] + df[c_60] + df[c_90]).sum())

            delinq_rows.append(
                {
                    "PropertyName": prop,
                    "Current": current_sum,
                    "Days30Plus": plus30_sum,
                }
            )

            print(f"✅ Delinq ok: {fname}")

        except Exception as e:
            print(f"❌ Delinq error: {p} | {e}")

    if not delinq_rows:
        raise Exception("No se pudo procesar ningún archivo Delinquent/Prepaid.")

    df_delinq = spark.createDataFrame(pd.DataFrame(delinq_rows))

    # =====================================================
    # 3) LIST FILES - BILLINGS
    # =====================================================
    df_files_bill = (
        spark.read.format("binaryFile")
        .option("pathGlobFilter", f"*{PATTERN_BILL}*.xlsx")
        .load(FOLDER_PATH)
        .withColumn("filename", F.element_at(F.split(F.col("path"), "/"), -1))
        .filter(~F.col("filename").isin(list(EXCLUDE)))
    )

    bill_paths = [r[0] for r in df_files_bill.select("path").collect()]
    print(f"Billing files: {len(bill_paths)}")

    # =====================================================
    # 4) PARSE BILLINGS (pandas)
    # =====================================================
    billing_rows = []

    for p in bill_paths:
        try:
            fname = filename_from_path(p)
            prop = prop_from_bill(fname)

            df_raw = pd.read_excel(p, sheet_name=0, header=None)

            if df_raw.shape[1] <= 24:
                raise Exception("No tiene columna index 24 para billings.")

            mask = df_raw.iloc[:, 1].astype(str).str.contains("Totals", na=False)
            totals = df_raw[mask]

            if totals.empty:
                billing_val = None
            else:
                val = totals.iloc[0, 24]
                mb = pd.to_numeric(val, errors="coerce")
                billing_val = float(mb) if pd.notna(mb) else None

            billing_rows.append(
                {
                    "PropertyName": prop,
                    "CurrentMonthBillings": billing_val,
                }
            )

            print(f"✅ Billing ok: {fname}")

        except Exception as e:
            print(f"❌ Billing error: {p} | {e}")

    if billing_rows:
        df_billings = spark.createDataFrame(pd.DataFrame(billing_rows))
    else:
        df_billings = spark.createDataFrame([], "PropertyName string, CurrentMonthBillings double")

    # =====================================================
    # 5) JOIN + CALCS
    # =====================================================
    df_final = (
        df_delinq
        .withColumn("PropertyName", F.upper(F.trim(F.col("PropertyName"))))
        .join(
            df_billings.withColumn("PropertyName", F.upper(F.trim(F.col("PropertyName")))),
            ["PropertyName"],
            "left",
        )
        .withColumn("RunDate", F.lit(RUN_DATE))
        .withColumn("CurrentMonthBillings", F.col("CurrentMonthBillings").cast("double"))
        .withColumn("Current", F.col("Current").cast("double"))
        .withColumn("Days30Plus", F.col("Days30Plus").cast("double"))
        .withColumn(
            "CurrentMonthBillingsNotCollected",
            F.when(
                (F.col("CurrentMonthBillings").isNull()) | (F.col("CurrentMonthBillings") == 0),
                F.lit(None).cast("double"),
            ).otherwise(
                F.round(F.col("Current") / F.col("CurrentMonthBillings"), 4)
            )
        )
    )

    df_final_clean = (
        df_final
        .select(
            F.to_date(F.col("RunDate")).alias("RunDate"),
            F.trim(F.col("PropertyName")).alias("PropertyName"),
            F.col("CurrentMonthBillingsNotCollected").cast("double").alias("CurrentMonthBillingsNotCollected"),
            F.col("CurrentMonthBillings").cast("double").alias("CurrentMonthBillings"),
            F.col("Current").cast("double").alias("Current"),
            F.col("Days30Plus").cast("double").alias("Days30Plus"),
        )
        .dropDuplicates(KEY_COLS)
    )

    rows_final = df_final_clean.count()
    print(f"📊 Filas df_final_clean: {rows_final}")
    if rows_final == 0:
        raise Exception("df_final_clean salió vacío; abortando escritura.")

    # =====================================================
    # 6) SAVE STG (overwrite)
    # =====================================================
    (
        df_final_clean.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable("stg_ar_collections")
    )
    print("✅ stg_ar_collections creado/actualizado")

    # =====================================================
    # 7) SAVE FINAL (append solo keys nuevas) - FIX TYPES vs tabla existente
    # =====================================================
    if not spark.catalog.tableExists("ar_collections"):
        (
            df_final_clean.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable("ar_collections")
        )
        print("✅ ar_collections creado (primera vez)")
    else:
        df_hist_keys = (
            spark.table("ar_collections")
            .select(
                F.to_date(F.col("RunDate")).alias("RunDate"),
                F.trim(F.col("PropertyName")).alias("PropertyName"),
            )
            .dropDuplicates(KEY_COLS)
        )

        df_to_append = df_final_clean.join(df_hist_keys, on=KEY_COLS, how="left_anti")
        to_append = df_to_append.count()
        print(f"🧾 Nuevos (RunDate, PropertyName) para append: {to_append}")

        if to_append > 0:
            target_df = spark.table("ar_collections")
            target_schema = target_df.schema

            # 1) casteá cada columna al tipo del destino (si existe)
            for field in target_schema.fields:
                col_name = field.name
                if col_name in df_to_append.columns:
                    df_to_append = df_to_append.withColumn(col_name, F.col(col_name).cast(field.dataType))

            # 2) reordená columnas EXACTO como la tabla destino
            target_cols = [f.name for f in target_schema.fields]
            df_to_append = df_to_append.select(*target_cols)

            # 3) append
            (
                df_to_append.write
                .format("delta")
                .mode("append")
                .saveAsTable("ar_collections")
            )
            print("✅ ar_collections actualizado (append sin duplicados por RunDate+PropertyName)")
        else:
            print("ℹ️ Nada nuevo que appendear (ya existían esos keys).")

StatementMeta(, 965c8fe1-ad75-4d47-803c-efabd09ce546, 7, Finished, Available, Finished, False)

# **<mark>Denies_Cancells ok</mark>**

In [6]:
def step5():
    # ==========================================================
    # Denies_Cancells
    # Resident Activity - Denied / Cancelled
    # Final table only (no STG tables)
    # ==========================================================

    from pyspark.sql import SparkSession
    from pyspark.sql.types import (
        StructType, StructField,
        StringType, DateType, DoubleType
    )
    from pyspark.sql import functions as F
    from notebookutils import mssparkutils
    import pandas as pd
    import re

    spark = SparkSession.builder.getOrCreate()

    # =========================
    # CONFIGURATION
    # =========================
    BASE_PATH = "file:/lakehouse/default/Files/xlsx_by_sheet"

    EXCLUDE_PROPERTIES = {
        "BBI LLC - Single Familiy",
        "First Property I LLC - Single Family",
        "Zoe LLC"
    }

    # =========================
    # LIST ALL EXCEL FILES
    # =========================
    all_files = [
        f.path
        for f in mssparkutils.fs.ls(BASE_PATH)
        if f.path.lower().endswith(".xlsx")
    ]

    if not all_files:
        raise Exception(f"No Excel files found in {BASE_PATH}")

    # =========================
    # FILTER TARGET FILES
    # =========================
    files = [
        p for p in all_files
        if re.search("resident_activity", p, re.I)
        and re.search("denied", p, re.I)
        and (re.search("cancel", p, re.I) or re.search("cancell", p, re.I))
    ]

    if not files:
        raise Exception("No matching Resident Activity Denied/Cancelled files found")

    # =========================
    # TARGET SCHEMA
    # =========================
    schema = StructType([
        StructField("Name", StringType(), True),
        StructField("Status", StringType(), True),
        StructField("BldgUnit", StringType(), True),
        StructField("CancelledDeniedDate", DateType(), True),
        StructField("CancelledDeniedReason", StringType(), True),
        StructField("LeasingConsultant", StringType(), True),
        StructField("DepositsOnHand", DoubleType(), True),
        StructField("LedgerBalance", DoubleType(), True),
        StructField("Property", StringType(), True),
        StructField("RunDate", DateType(), True),
    ])

    spark_dfs = []

    # =========================
    # HELPERS
    # =========================
    def safe_to_date(x):
        try:
            if pd.isna(x):
                return None
            return pd.to_datetime(x).date()
        except Exception:
            return None

    def safe_to_float(x):
        try:
            if pd.isna(x):
                return None
            if isinstance(x, str):
                x = x.replace(",", "").replace("$", "").strip()
                if x == "":
                    return None
            return float(x)
        except Exception:
            return None

    # =========================
    # PROCESS EACH FILE
    # =========================
    for path in files:
        file_name = path.split("/")[-1]

        try:
            xls = pd.ExcelFile(path)
            raw = xls.parse(xls.sheet_names[0], header=None)
        except Exception as e:
            print(f"⚠️ Skipping {file_name}: {e}")
            continue

        run_date = safe_to_date(raw.iloc[1, 0]) if raw.shape[0] > 1 else None

        header_vals = raw.head(10).astype(str).values.flatten().tolist()
        header_line = next((x for x in header_vals if " - " in x), None)
        fallback_property = file_name.split("_Resident")[0].replace("_", " ")
        property_val = header_line.split(" - ")[-1].strip() if header_line else fallback_property

        if property_val in EXCLUDE_PROPERTIES:
            print(f"⏭️ Skipping {file_name} (excluded Property: {property_val})")
            continue

        try:
            first_col = raw.iloc[:, 0].astype(str).tolist()
            header_idx = next(i for i, v in enumerate(first_col) if v.strip() == "Name")
        except Exception:
            print(f"⚠️ Header row 'Name' not found in {file_name}")
            continue

        df2 = raw.iloc[header_idx:].reset_index(drop=True)
        df2.columns = df2.iloc[0].astype(str)
        df2 = df2.iloc[1:].copy()

        name_col = df2.columns[0]
        df2 = df2[df2[name_col].notna()]
        df2 = df2[~df2[name_col].astype(str).str.strip().str.startswith("Total")]

        rename_map = {
            "Bldg/Unit": "BldgUnit",
            "Building/Unit": "BldgUnit",
            "Cancelled/Denied Date": "CancelledDeniedDate",
            "Cancel/Denied Date": "CancelledDeniedDate",
            "Cancelled/Denied\nDate": "CancelledDeniedDate",
            "Cancel/Denied\nDate": "CancelledDeniedDate",
            "Cancelled/Denied Reason": "CancelledDeniedReason",
            "Cancel/Denied Reason": "CancelledDeniedReason",
            "Cancelled/Denied\nReason": "CancelledDeniedReason",
            "Cancel/Denied\nReason": "CancelledDeniedReason",
            "Leasing Consultant": "LeasingConsultant",
            "Deposits on Hand": "DepositsOnHand",
            "Deposits On Hand": "DepositsOnHand",
            "Deposits\nOn Hand": "DepositsOnHand",
            "Ledger Balance": "LedgerBalance",
            "Ledger\nBalance": "LedgerBalance",
        }

        df2 = df2.rename(columns={k: v for k, v in rename_map.items() if k in df2.columns})

        for field in schema.fields:
            if field.name not in df2.columns:
                df2[field.name] = None

        df2["CancelledDeniedDate"] = df2["CancelledDeniedDate"].apply(safe_to_date)
        df2["DepositsOnHand"] = df2["DepositsOnHand"].apply(safe_to_float)
        df2["LedgerBalance"] = df2["LedgerBalance"].apply(safe_to_float)

        df2["Property"] = property_val
        df2["RunDate"] = run_date

        df2 = df2[[f.name for f in schema.fields]]

        spark_dfs.append(spark.createDataFrame(df2, schema=schema))
        print(f"✅ Processed: {file_name}")

    # =========================
    # UNION + SAVE
    # =========================
    if not spark_dfs:
        raise Exception("No valid files were processed")

    df_final = spark_dfs[0]
    for d in spark_dfs[1:]:
        df_final = df_final.unionByName(d, allowMissingColumns=True)

    # =========================
    # ADD bucket COLUMN
    # =========================
    reason_col = F.lower(F.trim(F.col("CancelledDeniedReason")))

    df_final = df_final.withColumn(
        "bucket",
        F.when(
            reason_col.contains("cancel")
            | reason_col.contains("changed their mind")
            | reason_col.contains("leased elsewhere"),
            F.lit("Cancelled")
        ).otherwise(F.lit("Denied"))
    )

    print(f"📊 denies_cancells rows: {df_final.count()}")

    (
        df_final.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable("denies_cancells")
    )

    df_final.show(20, truncate=False)

StatementMeta(, 965c8fe1-ad75-4d47-803c-efabd09ce546, 8, Finished, Available, Finished, False)

# **Effective_Rents**

In [7]:
def step6():
    # ============================================================
    # M -> PySpark (Fabric Lakehouse) | Effective Rents (Sheet2) - FINAL + HIST APPEND (NO DUPES)
    # ✅ stg_effective_rents = OVERWRITE (snapshot de hoy)
    # ✅ effective_rents = APPEND SOLO NUEVOS keys (RunDate, Property, Unit_Type)
    #    - si se corre 2 veces el mismo día no duplica
    # ============================================================

    import re
    import pandas as pd
    from pyspark.sql import SparkSession
    from pyspark.sql.functions import (
        col, lit, trim, lower, upper, regexp_replace, regexp_extract,
        when, sum as _sum, round, concat, current_date, date_sub
    )
    from notebookutils import mssparkutils

    spark = SparkSession.builder.getOrCreate()
    spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")

    BASE_PATH = "file:/lakehouse/default/Files/xlsx_by_sheet"

    EXCLUDE_FILES = {
        "BBI_LLC_-_Single_Familiy_Rent_Roll_Detail_-_Excel__Sheet2.xlsx",
        "First_Property_I_LLC_-_Single_Family_Rent_Roll_Detail_-_Excel__Sheet2.xlsx",
        "Zoe_LLC_Rent_Roll_Detail_-_Excel__Sheet2.xlsx",
    }

    # ================= LIST FILES =================
    paths = []
    for f in mssparkutils.fs.ls(BASE_PATH):
        if "Rent_Roll_Detail_-_Excel__Sheet2" in f.name and f.name not in EXCLUDE_FILES:
            paths.append(f.path)

    if not paths:
        raise Exception("❌ No Rent Roll files found")

    # ================= helpers =================
    def norm_header(x):
        if pd.isna(x):
            return ""
        s = str(x)
        s = s.replace("\n", " ").replace("\t", " ").replace('"', "")
        s = re.sub(r"\s+", " ", s).strip()
        return s

    def find_header_idx(pdf):
        for i in range(min(len(pdf), 200)):
            v = "" if pd.isna(pdf.iloc[i, 0]) else str(pdf.iloc[i, 0])
            if v.strip().lower() == "floorplan":
                return i
        return None

    def parse_property_from_pdf(pdf):
        for i in range(min(len(pdf), 80)):
            v = "" if pd.isna(pdf.iloc[i, 0]) else str(pdf.iloc[i, 0])
            vl = v.lower()
            if "heritage hill" in vl:
                if " - " in v:
                    return v.split(" - ", 1)[-1].strip()
                if "-" in v:
                    return v.split("-", 1)[-1].strip()
                return v.strip()
        return None

    def force_all_string(df_pd):
        for c in df_pd.columns:
            df_pd[c] = df_pd[c].astype("string")
        return df_pd

    def standardize_columns(data_pdf):
        cols = list(data_pdf.columns)
        normed = [norm_header(c).lower() for c in cols]

        def pick(pred):
            for original, n in zip(cols, normed):
                if pred(n):
                    return original
            return None

        rename_map = {}

        fp = pick(lambda n: n == "floorplan" or "floorplan" in n)
        if fp: rename_map[fp] = "Floorplan"

        uo = pick(lambda n: n == "units occupied" or ("unit" in n and "occup" in n))
        if uo: rename_map[uo] = "Units Occupied"

        al = pick(lambda n: n == "average leased" or ("average" in n and "leas" in n))
        if al: rename_map[al] = "Average Leased"

        nu = pick(lambda n: n == "# units" or n == "units" or (("unit" in n) and ("#" in n)))
        if nu: rename_map[nu] = "# Units"

        occp = pick(lambda n: n == "occupancy %" or ("occup" in n and "%" in n))
        if occp: rename_map[occp] = "Occupancy %"

        return data_pdf.rename(columns=rename_map)

    # ================= LOAD ALL FILES =================
    dfs = []

    for p in paths:
        try:
            pdf0 = pd.read_excel(p, sheet_name="Sheet2", header=None)

            header_idx = find_header_idx(pdf0)
            if header_idx is None:
                print(f"⚠️ skip (no Floorplan header): {p.split('/')[-1]}")
                continue

            prop = parse_property_from_pdf(pdf0)

            headers = [norm_header(x) for x in pdf0.iloc[header_idx].tolist()]
            data_pdf = pdf0.iloc[header_idx + 1:].copy()
            data_pdf.columns = headers

            data_pdf = standardize_columns(data_pdf)

            need = ["Floorplan", "# Units", "Average Leased", "Units Occupied", "Occupancy %"]
            for c in need:
                if c not in data_pdf.columns:
                    data_pdf[c] = None

            data_pdf["Property"] = prop
            data_pdf["SourceFile"] = p.split("/")[-1]
            data_pdf["RunDate"] = None

            data_pdf = force_all_string(data_pdf)

            dfs.append(spark.createDataFrame(data_pdf))
            print(f"✅ {p.split('/')[-1]} | Property={prop}")

        except Exception as e:
            print(f"⛔ Skipping {p.split('/')[-1]}: {e}")

    if not dfs:
        raise Exception("❌ All files failed")

    df_all = dfs[0]
    for d in dfs[1:]:
        df_all = df_all.unionByName(d, allowMissingColumns=True)

    # ✅ RunDate SIEMPRE = AYER
    df_all = df_all.withColumn("RunDate", date_sub(current_date(), 1))

    # ================= CLEAN FLOORPLAN + BEDROOM =================
    df_all = df_all.withColumn("Floorplan", col("Floorplan").cast("string"))

    df_all = df_all.withColumn(
        "Floorplan",
        trim(
            regexp_replace(
                regexp_replace(col("Floorplan"), r"[\u00A0\u2007\u202F]", " "),
                r"\s+", " "
            )
        )
    )

    df_all = df_all.filter(col("Floorplan").isNotNull() & (col("Floorplan") != ""))
    df_all = df_all.filter(~lower(col("Floorplan")).rlike(r"^\s*totals"))

    df_all = df_all.withColumn(
        "Bedroom",
        when(lower(col("Floorplan")).rlike(r"^\s*(studio|s)\b"), lit("0"))
        .otherwise(
            regexp_extract(
                regexp_replace(col("Floorplan"), r"^[^0-9]+", ""),
                r"^(\d+)",
                1
            )
        )
    )

    df_all = df_all.filter(col("Bedroom").isNotNull() & (trim(col("Bedroom")) != ""))
    df_all = df_all.filter(~upper(col("Bedroom")).isin(["T", "R", "P", "H", "F", "A"]))

    # ================= Unit Type (CONCAT FIX) =================
    df_all = df_all.withColumn(
        "Unit Type",
        when(col("Bedroom") == "0", lit("Studio Rent"))
        .otherwise(concat(col("Bedroom"), lit(" Bed Rent")))
    )

    # ================= PARSE NUMBERS =================
    def to_num(colname):
        return regexp_replace(
            regexp_replace(trim(col(colname).cast("string")), ",", ""),
            "%", ""
        ).cast("double")

    df_all = df_all.withColumn("Units Occupied", to_num("Units Occupied").cast("long"))
    df_all = df_all.withColumn("Average Leased", to_num("Average Leased"))

    df_all = df_all.filter(col("Average Leased").isNotNull() & (col("Average Leased") != 0))
    df_all = df_all.filter(col("Units Occupied").isNotNull() & (col("Units Occupied") > 0))
    df_all = df_all.filter(col("Property").isNotNull() & (trim(col("Property")) != ""))

    # ================= WEIGHTED AVG =================
    df_all = df_all.withColumn("SumProduct", col("Average Leased") * col("Units Occupied"))

    df_final = (
        df_all.groupBy("RunDate", "Property", "Unit Type")
              .agg(
                  _sum(col("SumProduct")).alias("SumProduct"),
                  _sum(col("Units Occupied")).alias("Units Occupied")
              )
              .withColumn("Effective Rent", round(col("SumProduct") / col("Units Occupied"), 0))
              .drop("SumProduct")
              .select("RunDate", "Property", "Unit Type", "Units Occupied", "Effective Rent")
    )

    # ================= DELTA SAFE COLUMN NAMES =================
    df_final_delta = (
        df_final
          .withColumnRenamed("Unit Type", "Unit_Type")
          .withColumnRenamed("Units Occupied", "Units_Occupied")
          .withColumnRenamed("Effective Rent", "Effective_Rent")
          .dropDuplicates(["RunDate", "Property", "Unit_Type"])
    )

    # ================= 1) STG = OVERWRITE =================
    (
        df_final_delta.write.format("delta")
          .mode("overwrite")
          .option("overwriteSchema", "true")
          .saveAsTable("stg_effective_rents")
    )
    print("✅ stg_effective_rents overwrite")

    # ================= 2) HIST = APPEND SOLO NUEVOS KEYS =================
    key_cols = ["RunDate", "Property", "Unit_Type"]

    target_exists = spark.catalog.tableExists("effective_rents")

    if not target_exists:
        (
            df_final_delta.write.format("delta")
              .mode("overwrite")
              .option("overwriteSchema", "true")
              .saveAsTable("effective_rents")
        )
        print("✅ effective_rents creado (primera vez)")
    else:
        hist = spark.table("effective_rents").select(*key_cols).dropDuplicates(key_cols)

        new_rows = (
            df_final_delta.alias("n")
            .join(hist.alias("h"), on=key_cols, how="left_anti")
        )

        new_count = new_rows.count()
        if new_count == 0:
            print("ℹ️ No hay nuevos keys. No append (evita duplicados).")
        else:
            (
                new_rows.write.format("delta")
                  .mode("append")
                  .saveAsTable("effective_rents")
            )
            print(f"✅ effective_rents append: {new_count} filas nuevas")

StatementMeta(, 965c8fe1-ad75-4d47-803c-efabd09ce546, 9, Finished, Available, Finished, False)

# **MOVE_INS ok**

In [8]:
def step7():
    # MOVE_INS (FIXED: no more LongType vs DoubleType + no more cannot determine type)
    # Strategy:
    # 1) Read each Excel with pandas
    # 2) Build a pandas DF with ONLY the columns we care about
    # 3) Force all pandas columns to STRING before createDataFrame (prevents merge/type inference issues)
    # 4) Union safely
    # 5) Cast in Spark to final types
    # 6) Save Delta

    from pyspark.sql import SparkSession
    from pyspark.sql.functions import col, lit
    from pyspark.sql.types import StringType, DateType, DoubleType
    from notebookutils import mssparkutils
    import pandas as pd
    import math

    spark = SparkSession.builder.getOrCreate()

    BASE_DIR = "file:/lakehouse/default/Files/xlsx_by_sheet"

    EXCLUDE_FILES = [
        "BBI_LLC_-_Single_Familiy_Resident_Activity_-_Excel__MOVE-INS.xlsx",
        "BBI_LLC_-_Single_Family_Resident_Activity_-_Excel_MOVE-INS.xlsx",
        "First_Property_I_LLC_-_Single_Family_Resident_Activity_-_Excel_MOVE-INS.xlsx",
        "Zoe_LLC_Resident_Activity_-_Excel_MOVE-INS.xlsx"
    ]

    TARGET_TABLE = "move_ins"

    print("\n================ Move Ins ================")

    # ================= LIST FILES =================
    files = [
        f.path
        for f in mssparkutils.fs.ls(BASE_DIR)
        if "resident_activity" in f.name.lower()
        and "move-ins" in f.name.lower()
        and f.name not in EXCLUDE_FILES
        and f.path.lower().endswith(".xlsx")
    ]

    if not files:
        raise Exception("No MOVE-INS files found")

    # ================= FINAL SCHEMA MAP =================
    final_cols = {
        "Name": "Name",
        "Status": "Status",
        "Bldg/Unit": "Bldg_Unit",
        "Lease Rent": "Lease_Rent",
        "Other Recurring Charges": "Other_Recurring_Charges",
        "Other Recurring Credits": "Other_Recurring_Credits",
        "Ad Source": "Ad_Source",
        "Move - In Date": "Move_In_Date",
        "Lease End Date": "Lease_End_Date",
        "Leasing Consultant": "Leasing_Consultant",
        "MOVE-INS": "MOVE_INS",
        "Property": "Property",
        "RunDate": "RunDate"
    }

    NUM_COLS = ["Lease Rent", "Other Recurring Charges", "Other Recurring Credits"]
    DATE_COLS = ["Move - In Date", "Lease End Date"]

    def _safe_str(x):
        if x is None or (isinstance(x, float) and math.isnan(x)) or pd.isna(x):
            return None
        return str(x)

    dfs = []

    # ================= PROCESS FILES =================
    for path in files:
        fname = path.split("/")[-1]
        try:
            pdf_raw = pd.read_excel(path, sheet_name=0, header=None)

            # ----- RunDate -----
            try:
                run_raw = pdf_raw.iloc[1, 0]
                run_date = pd.to_datetime(run_raw, errors="coerce").date()
            except Exception:
                run_date = None

            # ----- Property -----
            top10 = pdf_raw.head(10).astype(str).values.flatten()
            hdr_line = next((x for x in top10 if " - " in x), None)
            if hdr_line:
                prop = hdr_line.split(" - ", 1)[1]
            else:
                prop = fname.split("_Resident")[0].replace("_", " ")

            # ----- Find header row -----
            header_row_idx = None
            for i in range(len(pdf_raw)):
                if str(pdf_raw.iloc[i, 0]).strip() == "Name":
                    header_row_idx = i
                    break

            if header_row_idx is not None:
                pdf = pdf_raw.iloc[header_row_idx:].reset_index(drop=True)
                pdf.columns = pdf.iloc[0]
                pdf = pdf.iloc[1:]
            else:
                pdf = pdf_raw.copy()

            pdf.columns = [str(c).replace("\n", " ").replace("\r", " ").strip() for c in pdf.columns]

            # Ensure required columns exist in pandas
            for c in final_cols.keys():
                if c not in pdf.columns:
                    pdf[c] = None

            # Keep only what we need (plus Property/RunDate we add)
            pdf = pdf[list(final_cols.keys())].copy()

            # Filter out totals / blank names
            if "Name" in pdf.columns:
                pdf = pdf[pdf["Name"].notna()]
                pdf = pdf[~pdf["Name"].astype(str).str.startswith("Total", na=False)]

            # Parse numbers/dates in pandas (optional, but nice)
            for c in NUM_COLS:
                if c in pdf.columns:
                    pdf[c] = pd.to_numeric(pdf[c], errors="coerce")

            for c in DATE_COLS:
                if c in pdf.columns:
                    pdf[c] = pd.to_datetime(pdf[c], errors="coerce").dt.date

            # Add Property + RunDate
            pdf["Property"] = prop
            pdf["RunDate"] = str(run_date) if run_date else None

            # 🔥 Critical fix: Force ALL columns to STRING before Spark (prevents merge/type inference issues)
            for c in pdf.columns:
                pdf[c] = pdf[c].map(_safe_str)

            sdf = spark.createDataFrame(pdf)  # all strings now
            dfs.append(sdf)

            print(f"✅ {fname}")

        except Exception as e:
            print(f"❌ {fname} FAILED: {e}")

    if not dfs:
        raise Exception("All MOVE-INS files failed to process")

    # ================= UNION =================
    df = dfs[0]
    for d in dfs[1:]:
        df = df.unionByName(d, allowMissingColumns=True)

    # ================= FILTER BAD PROPERTIES =================
    df = df.filter(
        ~col("Property").isin(
            "First Property I LLC - Single Family",
            "Zoe LLC"
        )
    )

    # ================= RENAME TO FINAL =================
    for old, new in final_cols.items():
        if old not in df.columns:
            df = df.withColumn(old, lit(None).cast(StringType()))
        df = df.withColumnRenamed(old, new)

    df = df.select(list(final_cols.values()))

    # ================= CAST FINAL TYPES =================
    df_final = (
        df
        .withColumn("Name", col("Name").cast(StringType()))
        .withColumn("Status", col("Status").cast(StringType()))
        .withColumn("Bldg_Unit", col("Bldg_Unit").cast(StringType()))
        .withColumn("Ad_Source", col("Ad_Source").cast(StringType()))
        .withColumn("Leasing_Consultant", col("Leasing_Consultant").cast(StringType()))
        .withColumn("Property", col("Property").cast(StringType()))
        .withColumn("MOVE_INS", col("MOVE_INS").cast(StringType()))
        .withColumn("RunDate", col("RunDate").cast(StringType()))
        .withColumn("Move_In_Date", col("Move_In_Date").cast(DateType()))
        .withColumn("Lease_End_Date", col("Lease_End_Date").cast(DateType()))
        .withColumn("Lease_Rent", col("Lease_Rent").cast(DoubleType()))
        .withColumn("Other_Recurring_Charges", col("Other_Recurring_Charges").cast(DoubleType()))
        .withColumn("Other_Recurring_Credits", col("Other_Recurring_Credits").cast(DoubleType()))
    )

    # ================= SAVE =================
    (
        df_final.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable(TARGET_TABLE)
    )

    print(f"🎉 Table '{TARGET_TABLE}' written successfully")

StatementMeta(, 965c8fe1-ad75-4d47-803c-efabd09ce546, 10, Finished, Available, Finished, False)

# **Delinquent_Prepaid ok**

In [8]:
def step8():
    from pyspark.sql import SparkSession
    from pyspark.sql.functions import (
        col, lit, current_date, date_sub,
        trim, lower, coalesce, concat_ws, sha2,
        isnan, when, datediff, greatest
    )
    from pyspark.sql.types import (
        StringType, DoubleType, LongType, DateType,
        StructType, StructField
    )
    import pandas as pd
    import re
    import math
    from notebookutils import mssparkutils

    spark = SparkSession.builder.getOrCreate()

    # =========================
    # CONFIG
    # =========================
    BASE_PATH = "file:/lakehouse/default/Files/xlsx_by_sheet"
    SHEET2_TOKEN = "_Delinquent_and_Prepaid_-_Excel__Sheet2"
    SHEET1_TOKEN = "_Delinquent_and_Prepaid_-_Excel__Sheet1"

    HIST_TABLE = "deliquent_prepaid"

    KEY_COLS = ["resh_id", "lease_id", "bldg_unit"]

    ROW_HASH_COLS = [
        "resh_id","lease_id","bldg_unit","name","phone_number","email","status","move_in_out",
        "total_prepaid","total_delinquent","net_balance","current","days_30","days_60","days_90_plus",
        "prorate_credit","deposits_held","outstanding_deposit","late_count","nsf_count",
        "delinquency_comment","comment_date","property","total_delinquent_rent_s8",
        "days_since_comment_date"
    ]

    OUT_COLS = [
        "resh_id",
        "lease_id",
        "bldg_unit",
        "name",
        "phone_number",
        "email",
        "status",
        "move_in_out",
        "total_prepaid",
        "total_delinquent",
        "total_delinquent_rent_s8",
        "net_balance",
        "current",
        "days_30",
        "days_60",
        "days_90_plus",
        "prorate_credit",
        "deposits_held",
        "outstanding_deposit",
        "late_count",
        "nsf_count",
        "delinquency_comment",
        "comment_date",
        "days_since_comment_date",
        "property",
        "rundate",
        "record_hash",
        "row_hash"
    ]

    # =========================
    # HELPERS
    # =========================
    def join_key(x):
        if x is None or (isinstance(x, float) and math.isnan(x)) or pd.isna(x):
            return ""
        return str(x).strip().upper()

    def to_number(x):
        if x is None or (isinstance(x, float) and math.isnan(x)) or pd.isna(x):
            return None
        try:
            return float(x)
        except:
            t = str(x).replace(",", "").replace("$", "").strip()
            t = t.replace("(", "-").replace(")", "")
            try:
                return float(t)
            except:
                return None

    def find_header_row(df_raw, tokens, scan_rows=100):
        for i in range(min(scan_rows, len(df_raw))):
            row = [df_raw.iloc[i, j] for j in range(min(df_raw.shape[1], 50))]
            norm = []
            for v in row:
                if v is None or (isinstance(v, float) and math.isnan(v)) or pd.isna(v):
                    norm.append("")
                elif isinstance(v, str):
                    norm.append(v.strip())
                else:
                    norm.append(v)
            if all(tok in norm for tok in tokens):
                return i
        return -1

    def parse_banner_property(pdf_raw, fname):
        banner0 = None
        for i in range(min(3, len(pdf_raw))):
            v = pdf_raw.iloc[i, 0]
            if v is None or (isinstance(v, float) and math.isnan(v)) or pd.isna(v):
                continue
            banner0 = str(v)
            break

        banner_clean = banner0.replace("\r", "") if banner0 else ""
        lines = [l.strip() for l in banner_clean.split("\n") if l.strip()]

        prop_from_banner = ""
        if lines:
            parts = lines[0].split(" - ")
            if len(parts) >= 2:
                prop_from_banner = parts[1].strip()

        prop_from_file = fname.split("_Delinquent_and_Prepaid")[0].replace("_", " ")
        return prop_from_banner if prop_from_banner else prop_from_file

    # =========================
    # LIST FILES
    # =========================
    files_sheet2 = [
        f.path for f in mssparkutils.fs.ls(BASE_PATH)
        if SHEET2_TOKEN.lower() in f.name.lower() and f.path.lower().endswith(".xlsx")
    ]
    if not files_sheet2:
        raise Exception("❌ No matching Sheet2 files found.")

    files_sheet1 = [
        f.path for f in mssparkutils.fs.ls(BASE_PATH)
        if SHEET1_TOKEN.lower() in f.name.lower() and f.path.lower().endswith(".xlsx")
    ]

    # =========================
    # PROCESS SHEET2
    # =========================
    def process_sheet2(path):
        try:
            fname = path.split("/")[-1]
            pdf_raw = pd.read_excel(path, sheet_name=0, header=None)

            Property = parse_banner_property(pdf_raw, fname)

            header_idx = None
            for i in range(len(pdf_raw)):
                v = pdf_raw.iloc[i, 0]
                if v is None or (isinstance(v, float) and math.isnan(v)) or pd.isna(v):
                    continue
                if str(v).strip() == "Resh ID":
                    header_idx = i
                    break
            if header_idx is None:
                return None

            pdf = pdf_raw.iloc[header_idx:].copy()
            pdf.columns = pdf.iloc[0]
            pdf = pdf.iloc[1:]
            pdf.columns = [
                ("" if (c is None or (isinstance(c, float) and math.isnan(c)) or pd.isna(c)) else str(c).strip())
                for c in pdf.columns
            ]

            need = ["Resh ID", "Lease ID", "Bldg/Unit", "Name", "Phone Number", "Email", "Status", "Move-In/Out",
                    "Total Prepaid", "Total Delinquent", "Net Balance", "Current", "30 Days", "60 Days", "90+ Days",
                    "Prorate Credit", "Deposits Held", "Outstanding Deposit", "#Late", "#NSF", "Delinquency Comment",
                    "Comment Date"]
            for n in need:
                if n not in pdf.columns:
                    pdf[n] = None

            pdf = pdf[pdf["Resh ID"].notna() & pdf["Lease ID"].notna() & pdf["Bldg/Unit"].notna()]

            num_cols = [
                "Total Prepaid","Total Delinquent","Net Balance","Current",
                "30 Days","60 Days","90+ Days","Prorate Credit",
                "Deposits Held","Outstanding Deposit"
            ]
            for c in num_cols:
                pdf[c] = pd.to_numeric(pdf[c], errors="coerce")

            for c in ["Resh ID","Lease ID","#Late","#NSF"]:
                pdf[c] = pd.to_numeric(pdf[c], errors="coerce")

            pdf["Comment Date"] = pd.to_datetime(pdf["Comment Date"], errors="coerce").dt.date

            # ✅ FIX: convertir a string para que sobreviva el sanitize
            pdf["Comment Date"] = pdf["Comment Date"].apply(
                lambda x: x.strftime("%Y-%m-%d") if pd.notna(x) and hasattr(x, "strftime") else None
            )

            pdf["Property"] = Property
            pdf["JoinName"] = pdf["Name"].apply(join_key)

            return pdf
        except:
            return None

    # =========================
    # PROCESS SHEET1 (Rent - S8 map)
    # =========================
    def process_sheet1_s8_map(path):
        try:
            fname = path.split("/")[-1]
            pdf_raw = pd.read_excel(path, sheet_name=0, header=None)

            Property = parse_banner_property(pdf_raw, fname)

            hdr = find_header_row(df_raw=pdf_raw, tokens=["Name", "Code Description"], scan_rows=100)
            if hdr == -1:
                return None

            t = pdf_raw.iloc[hdr:].copy()
            t.columns = t.iloc[0]
            t = t.iloc[1:]
            t.columns = [
                ("" if (c is None or (isinstance(c, float) and math.isnan(c)) or pd.isna(c)) else str(c).strip())
                for c in t.columns
            ]

            if "Name" not in t.columns or "Code Description" not in t.columns or "Total Delinquent" not in t.columns:
                return None

            t["Property"] = Property
            t["JoinName"] = t["Name"].apply(join_key)

            t = t[t["Code Description"].astype(str).str.contains("Rent - S8", case=False, na=False)]
            t["S8Delinquent"] = t["Total Delinquent"].apply(to_number)

            g = (
                t.groupby(["Property","JoinName"])["S8Delinquent"]
                .sum(min_count=1)
                .reset_index()
                .rename(columns={"S8Delinquent":"TotalDelinquent_RentS8"})
            )
            return g
        except:
            return None

    # =========================
    # BUILD PANDAS
    # =========================
    pdfs2 = [x for f in files_sheet2 if (x := process_sheet2(f)) is not None]
    if not pdfs2:
        raise Exception("❌ Found Sheet2 files but none could be parsed.")
    pdf = pd.concat(pdfs2, ignore_index=True)

    maps = [x for f in files_sheet1 if (x := process_sheet1_s8_map(f)) is not None]
    if maps:
        pdf = pdf.merge(pd.concat(maps, ignore_index=True), how="left", on=["Property","JoinName"])
    else:
        pdf["TotalDelinquent_RentS8"] = None

    # =========================
    # FINAL RENAME
    # =========================
    pdf.rename(columns={
        "Resh ID":"resh_id",
        "Lease ID":"lease_id",
        "Bldg/Unit":"bldg_unit",
        "Name":"name",
        "Phone Number":"phone_number",
        "Email":"email",
        "Status":"status",
        "Move-In/Out":"move_in_out",
        "Total Prepaid":"total_prepaid",
        "Total Delinquent":"total_delinquent",
        "TotalDelinquent_RentS8":"total_delinquent_rent_s8",
        "Net Balance":"net_balance",
        "Current":"current",
        "30 Days":"days_30",
        "60 Days":"days_60",
        "90+ Days":"days_90_plus",
        "Prorate Credit":"prorate_credit",
        "Deposits Held":"deposits_held",
        "Outstanding Deposit":"outstanding_deposit",
        "#Late":"late_count",
        "#NSF":"nsf_count",
        "Delinquency Comment":"delinquency_comment",
        "Comment Date":"comment_date",
        "Property":"property"
    }, inplace=True)

    pdf.columns = [re.sub(r"[^a-zA-Z0-9_]", "_", str(c).lower()) for c in pdf.columns]

    def _sanitize_cell(x):
        if x is None or (isinstance(x, float) and math.isnan(x)) or pd.isna(x):
            return None
        if isinstance(x, (dict, list, tuple, set)):
            return str(x)
        return x

    pdf = pdf.apply(lambda s: s.map(_sanitize_cell))

    schema = StructType([StructField(c, StringType(), True) for c in pdf.columns])
    df = spark.createDataFrame(pdf, schema=schema)

    # =========================
    # CASTS + RUNDATE (AYER)
    # =========================
    df = (
        df
        .withColumn("resh_id", col("resh_id").cast(LongType()))
        .withColumn("lease_id", col("lease_id").cast(LongType()))
        .withColumn("bldg_unit", col("bldg_unit").cast(StringType()))
        .withColumn("name", col("name").cast(StringType()))
        .withColumn("phone_number", col("phone_number").cast(StringType()))
        .withColumn("email", col("email").cast(StringType()))
        .withColumn("status", col("status").cast(StringType()))
        .withColumn("move_in_out", col("move_in_out").cast(StringType()))
        .withColumn("total_prepaid", col("total_prepaid").cast(DoubleType()))
        .withColumn("total_delinquent", col("total_delinquent").cast(DoubleType()))
        .withColumn("total_delinquent_rent_s8", col("total_delinquent_rent_s8").cast(DoubleType()))
        .withColumn("net_balance", col("net_balance").cast(DoubleType()))
        .withColumn("current", col("current").cast(DoubleType()))
        .withColumn("days_30", col("days_30").cast(DoubleType()))
        .withColumn("days_60", col("days_60").cast(DoubleType()))
        .withColumn("days_90_plus", col("days_90_plus").cast(DoubleType()))
        .withColumn("prorate_credit", col("prorate_credit").cast(DoubleType()))
        .withColumn("deposits_held", col("deposits_held").cast(DoubleType()))
        .withColumn("outstanding_deposit", col("outstanding_deposit").cast(DoubleType()))
        .withColumn("late_count", col("late_count").cast(LongType()))
        .withColumn("nsf_count", col("nsf_count").cast(LongType()))
        .withColumn("delinquency_comment", col("delinquency_comment").cast(StringType()))
        .withColumn("comment_date", col("comment_date").cast(DateType()))
        .withColumn("property", col("property").cast(StringType()))
        .withColumn("rundate", date_sub(current_date(), 1))
    )

    # =========================
    # NEW: days_since_comment_date
    # =========================
    df = df.withColumn(
        "days_since_comment_date",
        when(
            col("comment_date").isNull(),
            lit(None).cast(LongType())
        ).otherwise(
            greatest(
                datediff(current_date(), col("comment_date")),
                lit(0)
            )
        )
    )

    # =========================
    # FIX: NaN/Infinity -> NULL
    # =========================
    POS_INF = float("inf")
    NEG_INF = float("-inf")
    double_cols = [c for c, t in df.dtypes if t in ("double", "float")]
    for c in double_cols:
        df = df.withColumn(
            c,
            when(
                col(c).isNull() | isnan(col(c)) | (col(c) == lit(POS_INF)) | (col(c) == lit(NEG_INF)),
                lit(None).cast(DoubleType())
            ).otherwise(col(c))
        )

    # asegurar columnas de row_hash
    for c in ROW_HASH_COLS:
        if c not in df.columns:
            df = df.withColumn(c, lit(None))

    # =========================
    # HASHES
    # =========================
    df = df.withColumn(
        "record_hash",
        sha2(
            concat_ws(
                "||",
                *[lower(trim(coalesce(col(c).cast("string"), lit("")))) for c in KEY_COLS]
            ),
            256
        )
    )

    df = df.withColumn(
        "row_hash",
        sha2(
            concat_ws(
                "||",
                *[lower(trim(coalesce(col(c).cast("string"), lit("")))) for c in ROW_HASH_COLS]
            ),
            256
        )
    )

   # =========================
    # SELECT + overwrite HIST
    # =========================
    df_final = df.select(*OUT_COLS)

    # 👇 AGREGA ESTO AQUÍ
    print("🔍 comment_date no nulos antes de guardar:")
    df_final.filter(col("comment_date").isNotNull()).select("comment_date", "property").show(10)
    print(f"Total no nulos: {df_final.filter(col('comment_date').isNotNull()).count()}")

    (
        df_final.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable(HIST_TABLE)
    )

StatementMeta(, 75c4fbe7-7dca-4573-bfad-a597050382f5, 12, Finished, Available, Finished, False)

# **Leasing_desk_scores ok**

In [11]:
def step9():
    from pyspark.sql import SparkSession, functions as F
    from notebookutils import mssparkutils
    import pandas as pd
    import re, math

    spark = SparkSession.builder.getOrCreate()
    spark.sql("USE One_Site_LAH_DF2")   # ← setea el contexto

         # ← sin "dbo."

    BASE_DIR = "file:/lakehouse/default/Files/xlsx_by_sheet"
    FILE_REGEX = r"Applicant_Detail__Applicant_Detail.*\.xlsx$"
    SHEET_NAME = "Applicant_Detail"
    TARGET_TABLE = "Leasing_Desk"

    def clean_and_dedupe_columns(cols):
        out, seen = [], {}
        for i, c in enumerate(cols, start=1):
            if c is None or (isinstance(c, float) and math.isnan(c)):
                c = f"Column{i}"
            c = str(c).strip()
            if c == "" or c.lower() == "nan" or c.lower().startswith("unnamed"):
                c = f"Column{i}"
            if c in seen:
                seen[c] += 1
                c = f"{c}_{seen[c]}"
            else:
                seen[c] = 1
            out.append(c)
        return out

    def delta_safe(name: str) -> str:
        s = str(name).strip()
        s = re.sub(r"[ ,;{}\(\)\n\t=]", "_", s)
        s = re.sub(r"[^0-9A-Za-z_]", "", s)
        s = re.sub(r"_+", "_", s).strip("_")
        if s == "" or s[0].isdigit():
            s = f"col_{s}" if s else "col"
        return s

    def to_safe_date_string(series: pd.Series) -> pd.Series:
        dt = pd.to_datetime(series, errors="coerce")
        return dt.dt.strftime("%Y-%m-%d")

    files = [f.path for f in mssparkutils.fs.ls(BASE_DIR) if re.search(FILE_REGEX, f.name, re.I)]
    if not files:
        raise Exception("❌ No encontré archivos Applicant_Detail en la carpeta")

    dfs = []

    for path in files:
        try:
            fname = path.split("/")[-1]

            pdf = pd.read_excel(path, sheet_name=SHEET_NAME, header=None)

            pdf = pdf[pdf.iloc[:, 0].notna()]
            pdf = pdf[pdf.iloc[:, 0].astype(str) != "Unnamed: 0"]

            header = list(pdf.iloc[0].values)
            pdf = pdf.iloc[1:].reset_index(drop=True)
            pdf.columns = clean_and_dedupe_columns(header)

            pdf = pdf.dropna(axis=1, how="all")

            drop_cols = ["Applicant_Detail.xls", "Applicant Detail", "Column13", "Column8"]
            pdf = pdf.drop(columns=[c for c in drop_cols if c in pdf.columns], errors="ignore")

            date_cols = ["Movein Date", "Decision Date", "App Result Date", "App Submission Date"]
            for c in date_cols:
                if c in pdf.columns:
                    pdf[c] = to_safe_date_string(pdf[c])

            for c in pdf.columns:
                if pdf[c].apply(lambda x: isinstance(x, (dict, list, tuple))).any():
                    pdf[c] = pdf[c].astype(str)

            sdf = spark.createDataFrame(pdf)

            sdf = (
                sdf.withColumn("SourceFile", F.lit(fname))
                   .withColumn("RunDate", F.current_date())
            )

            dfs.append(sdf)
            print(f"✅ Loaded: {fname}")

        except Exception as e:
            print(f"⚠️ Skipping {path}: {e}")

    if not dfs:
        raise Exception("❌ Todos los archivos fallaron")

    df = dfs[0]
    for d in dfs[1:]:
        df = df.unionByName(d, allowMissingColumns=True)

    orig_cols = df.columns
    safe_cols = []
    seen = {}

    for c in orig_cols:
        s = delta_safe(c)
        if s in seen:
            seen[s] += 1
            s = f"{s}_{seen[s]}"
        else:
            seen[s] = 1
        safe_cols.append(s)

    for o, s in zip(orig_cols, safe_cols):
        if o != s:
            df = df.withColumnRenamed(o, s)

    safe_date_cols = [delta_safe(x) for x in ["Movein Date", "Decision Date", "App Result Date", "App Submission Date"]]
    for c in safe_date_cols:
        if c in df.columns:
            df = df.withColumn(c, F.to_date(F.col(c)))

    print("🧾 Columnas finales antes de guardar:", df.columns)

    (
        df.write
          .mode("overwrite")
          .option("overwriteSchema", "true")
          .format("delta")
          .saveAsTable(TARGET_TABLE)
    )

    # Limpieza final de LDSScore:
    # borra filas donde LDSScore sea 'No Tradelines', 'No Records'
    # o cualquier valor que no se pueda convertir a número
    spark.sql(f"""
    DELETE FROM {TARGET_TABLE}
    WHERE lower(trim(cast(LDSScore as string))) IN ('no tradelines', 'no records')
       OR cast(trim(cast(LDSScore as string)) as double) IS NULL
    """)

    spark.catalog.refreshTable(TARGET_TABLE)

    print(f"✅ Tabla guardada y limpiada: {TARGET_TABLE}")

StatementMeta(, 965c8fe1-ad75-4d47-803c-efabd09ce546, 13, Finished, Available, Finished, False)

# **Final_Output**

In [12]:
# =========================
# PIPELINE EXECUTION + SUMMARY
# =========================

import traceback

STEP_RESULTS = []

def run_step(name, fn):
    print(f"\n================ {name} ================")
    try:
        fn()
        STEP_RESULTS.append((name, "SUCCESS", None))
        print(f"✅ {name} OK")
    except Exception as e:
        STEP_RESULTS.append((name, "FAILED", str(e)))
        print(f"❌ {name} FAILED")
        print(traceback.format_exc())

# ---------- Run steps ----------
run_step("Step 1", step1)
run_step("Step 2", step2)
run_step("Step 3", step3)
run_step("Step 4", step4)
run_step("Step 5", step5)
run_step("Step 6", step6)
run_step("Step 7", step7)
run_step("Step 8", step8)
run_step("Step 9", step9)

# ---------- Summary ----------
print("\n================ PIPELINE SUMMARY ================")
for name, status, err in STEP_RESULTS:
    if status == "SUCCESS":
        print(f"✅ {name}")
    else:
        print(f"❌ {name}: {err}")

failures = [s for s in STEP_RESULTS if s[1] == "FAILED"]

# IMPORTANT: fail the notebook if anything failed (so Fabric doesn't lie)
if failures:
    raise Exception(f"{len(failures)} step(s) failed. First error: {failures[0][2]}")
else:
    print("\n🎉 All steps completed successfully.")

StatementMeta(, 965c8fe1-ad75-4d47-803c-efabd09ce546, 14, Finished, Cancelled, Cancelled, False)

PRIMARY required pattern: SpecificDate_SystemDate_3_25_2026*.zip
SECONDARY required patterns (today):
 - OnDemand_*20260325*.zip
 - SpecificDate_SystemDate_3_25_2026*.zip
ENFORCED proceed/fail folder: /os_home/ReportGroups/Fabric Reports
Upload will include PRIMARY + SECONDARY.

TODAY zips found for upload:
 - /os_home/ReportGroups/Fabric Reports/SpecificDate_SystemDate_3_25_20262_20_13AM20260325_072135328_Fabric_Reports.zip
 - /os_home/ReportGroups/Fabric Reports 2/OnDemand_03_25_2610_00_01AM20260325_100303929_Fabric_Reports_2.zip

Resolved SharePoint target (site+path):
 - siteId: heritagehillpm.sharepoint.com,29a6fbdc-624b-49de-8793-ac2536077dcd,672ac0c8-53ee-4cc2-9300-db705762b1ed
 - driveId: b!3PumKUti3kmHk6wlNgd9zcjAKmfuU8JMkwDbcFdise3Lu_bB5dauSaHtK9pjH4-1
 - folderItemId: 014IYA4E2VM3EDF7W3KRFZ7MNV5BULYKP2
 - folderPath: Fabric Data Drop/OneSite Reports

Downloading: /os_home/ReportGroups/Fabric Reports/SpecificDate_SystemDate_3_25_20262_20_13AM20260325_072135328_Fabric_Reports.

# **Pruebas**

In [5]:
df = spark.sql("SELECT * FROM One_Site_LAH_DF2.deliquent_prepaid LIMIT 1000")
display(df)

StatementMeta(, b194fc34-95d0-4ae3-aefd-c56002cf4d48, 7, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 75a383da-fce5-449d-a66e-2e3c600173cf)

In [ ]:
%%sql
RESTORE TABLE ar_collections TO VERSION AS OF 1;


StatementMeta(, fdc42e98-98fe-4b59-b10e-25b7d9e33af1, -1, Cancelled, , Cancelled)